# Outros

In [ ]:
## TESTE
import re
import pandas as pd

# ==============================================================================
# 1. DICIONÁRIOS DE TERMOS (SUA BASE VALIDADA + NCM COMPLETO)
# ==============================================================================

# Exclusões Críticas
EXCL_PRIORITARIA = 'ENTREGA|TAXA|FRETE|COMPLEMENTO DE VALOR|LOTE MISTO|MERCADORIA MISTA|SERVICO|VALOR ADICIONAL|AJUSTE'
EXCL_MATERIAIS = 'VELUDO|PLASTICO|METAL|ALUMINIO|FERRO|ACO|VIDRO|BLINDEX|FUME|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO'
EXCL_MOVEIS = 'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA|COZINHA|PLANEJADA|PLANEJADOS'
EXCL_DECORACAO = 'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA|APLIQUE|RECORTE|LASER'
EXCL_SERVICOS = 'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|MONTAGEM|INSTALACAO'

# Validadores e Espécies
TERMOS_ESPECIES = 'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA'
TERMOS_PROCESSAMENTO = 'BENEFICIADA|SERRADA|SERRADO|VIGA|TABUA|TABUAS|TRONCO|FLORESTAL|CASCA|TORAS|TORA|CAVACO|LENHA|RESIDUO|BIOMASSA|MOINHA|18MM|1100X2200|FSC|CERTIFICADA|CERTIFICADO'

# ==============================================================================
# 2. ESTRUTURA HIERÁRQUICA DE CATEGORIAS
# ==============================================================================

categorias_regex = [
    # A. BLOQUEIO DE OUTLIERS (Prioridade Máxima)
    ('OUTLIER__FINANCEIRO', rf'.*({EXCL_PRIORITARIA}).*'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    
    # B. ROTULADOS ANTES
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # C. NCM 44.02 - CARVÃO VEGETAL (Com suas exclusões validadas)
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO)).*'),

    # D. NCM 44.18 - PORTAS E JANELAS (Com suas exclusões validadas)
    ('44.18.PORTA_JANELA', rf'^(?!.*({EXCL_MATERIAIS}|PVC|APLIQUE|RECORTE|LASER|MINIATURA|BONECA|POLLY|BRINQUEDO|ARTESANATO|ENFEITE|DECORACAO|GELADEIRA|CORTINA|PERSIANA|TELA|MOSQUITEIRO)(?!.*(MADEIRA|MAD|MDF|{TERMOS_ESPECIES})))(?=.*(PORTA|JANELA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO))(?=.*(MADEIRA|MAD|MDF|{TERMOS_ESPECIES}|210|80X|70X|90X)).*'),

    # E. NCM 44.11/44.12 - PAINÉIS E MDF (Com suas exclusões validadas)
    ('44.11.MDF_PAINEL', rf'^(?!.*(PLASTIFICADO|{EXCL_MOVEIS}))(?=.*(MDF|MDP|OSB|HDF|COMPENSADO|ARAUCO|GREENPLAC))(?=.*(18MM|15MM|3MM|1100X2200|MADEIRA|MAD|{TERMOS_ESPECIES})).*'),

    # F. NCM 44.15 - PALETES E EMBALAGENS (Com suas exclusões validadas)
    ('44.21.PALETE_PALLET', rf'^(?=.*(PALETE|PALLET))(?!.*({EXCL_SERVICOS})).*'),

    # G. NCM 44.03/44.07 - MATÉRIA-PRIMA E SERRADOS (Inclusão por Espécie/Processo)
    ('44.03.MADEIRA_BRUTA_SERRADA', rf'.*({TERMOS_ESPECIES}|{TERMOS_PROCESSAMENTO}).*'),

    # H. NCM 44.21 - ARTEFATOS E UTENSÍLIOS
    ('44.21.ARTEFATOS_MADEIRA', rf'^(?!.*({EXCL_MATERIAIS})(?!.*(MADEIRA|MAD|MDF|BAMBU|{TERMOS_ESPECIES})))(?=.*(CABIDE|ESPETO|ESPETINHO|CAKE BOARD|BANDEJA|CABO|FERRAMENTA|CAIXA|ADORNOS|PRENDEDOR|CAIXOTE|ESTRADO|PALITO|CEPO))(?=.*(MADEIRA|MAD|MDF|BAMBU|CERTIFICADA|FSC|{TERMOS_ESPECIES})).*'),

    # I. OUTROS SUB-CAPÍTULOS (44.01 a 44.21 - Cobertura Genérica)
    ('44.01.LENHA_RESIDUOS', rf'^(?=.*(LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA))(?!.*({EXCL_MATERIAIS})).*'),
    ('44.04.ARCOS_ESTACAS', rf'^(?=.*(ARCO|ESTACA|MOIRAO|POSTE))(?=.*(MADEIRA)).*'),
    ('44.09.PERFILADOS', rf'^(?=.*(MOLDURA|RODAPE|TACO|LAMBRI))(?=.*(MADEIRA)).*'),
    ('44.17.FERRAMENTAS', rf'^(?=.*(FERRAMENTA|CABO|PÁ|ENXADA))(?=.*(MADEIRA)).*'),
    ('44.19.UTENSILIOS_MESA', rf'^(?=.*(PRATO|COPO|TIGELA|COLHER))(?=.*(MADEIRA|BAMBU)).*'),

    # J. CATEGORIA DE REVISÃO E FECHAMENTO
    ('AMBIGUOS_REVISAR', rf'.*(PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTROS', r'.*')
]

# ==============================================================================
# 3. PROCESSAMENTO E EXPORTAÇÃO
# ==============================================================================

def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

# Aplicação
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)
# --- AJUSTE FINAL NA SEPARAÇÃO DOS ARQUIVOS (TAREFA 4) ---

# Lista exata de categorias que você considera como "CAPÍTULO 44" para a base de treino
categorias_cap_44 = [
    'ROTULADOS ANTES', 
    '44.18.PORTA_JANELA', 
    '44.02.CARVAO_VEGETAL', 
    '44.21.PALETE_PALLET', 
    '44.11.MDF_PAINEL', 
    '44.03.MADEIRA_BRUTA_SERRADA',
    '44.01.LENHA_RESIDUOS',
    '44.21.ARTEFATOS_MADEIRA',
    '44.04.ARCOS_ESTACAS',
    '44.09.PERFILADOS',
    '44.17.FERRAMENTAS',
    '44.19.UTENSILIOS_MESA'
]

# Lista exata de categorias para ELIMINADOS
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'AMBIGUOS_REVISAR']

# Filtragem
df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

# Exportação com os nomes de arquivo exatos que você pediu (_pre)
df_cap_44.to_csv('base_treino_capitulo_44_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre.csv', sep=';', index=False, encoding='utf-8-sig')

# Print formatado conforme sua solicitação
print(f"1. CAPÍTULO 44: {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:        {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS (Lixo/Outros Cap): {len(df_eliminados):>10,}".replace(",", "."))
print(f"TOTAL DE ITENS PROCESSADOS:      {len(df_pandas):>10,}".replace(",", "."))


1. CAPÍTULO 44:    896.805
2. OUTROS:           846.510
3. ELIMINADOS (Lixo/Outros Cap):     12.060
TOTAL DE ITENS PROCESSADOS:       1.755.375

7m31s de execução

In [ ]:
# 1. Adicione os novos termos de exclusão específicos para PORTA
termos_excl_porta_lixo = 'RETRATO|COPO|CHAVE|TEMPERO|GUARDANAPO|JOIA|CAPSULA|TOALHA|ESPETO|FRIO|XICARA|HIGIENICO|ESTEIRA|BANDEJA|SUPORTE'
termos_excl_porta_mat = 'FERRO|ALUMINIO|PVC|VIDRO|ACO'
termos_valid_madeira = 'MADEIRA|MDF|MDP|ANGELIM|PINUS|EUCALIPTO|MOGNO|TAUARI|CEDRO|VIROLA|CURUPIXA|FRISADA|LISA|MACICA|PIVOTANTE|CAMARAO|SEMISSOLIDA|PRANCHETA|BATENTE|CAIXILHO|210|210X|80X|70X|90X'

# 2. Atualize a lista de categorias_regex
categorias_regex = [
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    
    # NOVA CATEGORIA: 44.18.PORTA_MADEIRA (Inserida antes das genéricas)
    # Lógica: (Não é lixo) E (Não é material concorrente sem madeira) E (Tem a palavra PORTA) E (Tem validador de madeira)
    ('44.18.PORTA_MADEIRA', rf'^(?!.*({termos_excl_porta_lixo}))(?!.*({termos_excl_porta_mat})(?!.*(MADEIRA|MDF|MDP)))(?=.*PORTA)(?=.*({termos_valid_madeira})).*'),
    
    ('44.18.JANELA_MADEIRA', rf'^(?=.*JANELA)(?!.*({termos_excl_janela})).*'),
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*({termos_excl_carvao})).*'),
    ('44.21.PALETE_PALLET', rf'^(?=.*(PALETE|PALLET))(?!.*({termos_excl_palete})).*'),
    ('44.11.MDF_PAINEL', rf'^(?=.*(MDF|MDP|OSB|CHAPATEX|HDF))(?!.*({termos_excl_mdf})).*'),
    ('AMBIGUOS_EXCLUSAO', rf'.*({termos_ambiguos_exclusao}).*'),
    ('NOMES_AMBÍGUOS', rf'^(?=.*({termos_ambiguos_inclusao}))(?=.*(MADEIRA|MDF|MDP|PINUS|MM|X)).*'),
    ('44.03.MADEIRA_BRUTA', r'(MADEIRA.*(BRUTA|TORA|CELULOSE)|EUCALIPTO.*(TORA|CELULOSE)|PINUS.*BRUTA)'),
    
    # Ajustei esta categoria para não conflitar com a nova regra de PORTA acima
    ('44.18.OBRAS_CARPINTARIA', r'(FOLHA DE PORTA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO)'),
    
    ('44.01.CAVACO_RESIDUO', r'(CAVACO DE EUCALIPTO|LENHA|CAVACO|RESIDUO|GRANULADO|BIOMASSA|SERRAGEM|PO DE MADEIRA)'),
    ('44.21.OUTRAS_OBRAS', r'(PALITO|CEPO|ESTRADO|CAIXOTE)'),
    ('OUTROS', r'.*')
]


def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

In [ ]:
# --- 1. LISTA AUTOMÁTICA DE CATEGORIAS DO CAPÍTULO 44 ---
# Isso garante que todas as novas categorias (44.01, 44.18.10, etc.) sejam incluídas automaticamente
categorias_cap_44 = [
    cat for cat in df_pandas['categoria_regex'].unique() 
    if cat.startswith('44.') or cat == 'ROTULADOS ANTES'
]

# --- 2. LISTA DE CATEGORIAS ELIMINADAS ---
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'AMBIGUOS_REVISAR']

# --- 3. FILTRAGEM DOS DATAFRAMES ---
df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

# --- 4. EXPORTAÇÃO DOS ARQUIVOS (_pre_r) ---
df_cap_44.to_csv('base_treino_capitulo_44_pre_r.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre_r.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre_r.csv', sep=';', index=False, encoding='utf-8-sig')

# --- 5. PRINT DE RESULTADOS ---
print(f"1. CAPÍTULO 44 (Total Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                       {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS (Lixo/Outros Cap): {len(df_eliminados):>10,}".replace(",", "."))
print("-" * 40)
print(f"TOTAL DE ITENS PROCESSADOS:      {len(df_pandas):>10,}".replace(",", "."))


1. CAPÍTULO 44:    896.805
2. OUTROS:           846.510
3. ELIMINADOS (Lixo/Outros Cap):      1.450
TOTAL DE ITENS PROCESSADOS:       1.755.375

In [ ]:
# 1. DICIONÁRIOS DE TERMOS E EXCLUSÕES

EXCL_MATERIAIS = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO|GRANITO|MARMORE|PEDRA'
EXCL_MOVEIS = 'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA|COZINHA|PLANEJADA|PLANEJADOS'
EXCL_DECORACAO = 'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA|APLIQUE|RECORTE|LASER'
EXCL_SERVICOS = 'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|FRETE|MONTAGEM|INSTALACAO|TAXA|ENTREGA|AJUSTE'
TERMOS_ESPECIES = 'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA'

# 2. ESTRUTURA DE CATEGORIAS
categorias_regex = [
    # A. BLOQUEIO DE OUTLIERS
    ('OUTLIER__FINANCEIRO', rf'.*({EXCL_SERVICOS}|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO)'),
    
    # B. ROTULADOS ANTES
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # C. NCM 44.01 a 44.06 (Matéria-Prima e Resíduos)
    ('44.01.LENHA_RESIDUOS', r'^(?=.*(LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA))(?!.*(ALUMINIO|FERRO|METAL|PVC)).*'),
    ('44.02.CARVAO_VEGETAL', r'^(?=.*(CARVAO|CARVÃO|MOINHA))(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE)).*'),
    ('44.03.MADEIRA_BRUTA', r'^(?=.*(MADEIRA.*(BRUTA|EM BRUTO)|TORA|TORO|RONCO))(?!.*(SERRADA|APARELHADA|BENEFICIADA)).*'),
    ('44.04.ARCOS_ESTACAS', r'^(?=.*(ARCO|ESTACA|MOIRAO|POSTE))(?=.*(MADEIRA)).*'),
    ('44.05.LA_FARINHA', r'^(?=.*(LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA))(?!.*(MINERAL|VIDRO|SINTETICA)).*'),
    ('44.06.DORMENTES', r'^(?=.*(DORMENTE))(?=.*(MADEIRA|FERROVIA|TRILHO))(?!.*(CONCRETO|METAL)).*'),

    # D. NCM 44.07 a 44.09 (Madeira Processada)
    ('44.07.MADEIRA_SERRADA', rf'^(?=.*(TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE|{TERMOS_ESPECIES}))(?!.*(COMPENSADO|MDF|PAINEL)).*'),
    ('44.08.FOLHAS_FOLHEADOS', r'^(?=.*(FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA))(?=.*(6MM|6 MM|FINA|DELGADA)).*'),
    ('44.09.PERFILADOS', r'^(?=.*(MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR))(?=.*(MADEIRA|PINUS|EUCALIPTO)).*'),

    # E. NCM 44.10 a 44.12 (Painéis e Chapas)
    ('44.10.PAINEIS_PARTICULAS_OSB', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(OSB|MDP|PARTICULA|AGLOMERADO))(?=.*(MADEIRA|MAD)).*'),
    ('44.11.PAINEIS_FIBRAS_MDF', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(MDF|HDF|FIBRA|ARAUCO|GREENPLAC))(?=.*(18MM|15MM|3MM|MADEIRA|MAD)).*'),
    ('44.12.COMPENSADO', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(COMPENSADO|CONTRAPLACADO|LAMINADO))(?=.*(MADEIRA|MAD|PINUS|EUCALIPTO)).*'),

    # F. NCM 44.13 a 44.17 (Obras Diversas)
    ('44.13.MADEIRA_DENSIFICADA', r'^(?=.*(DENSIFICADA|ESTABILIZADA))(?=.*(MADEIRA)).*'),
    ('44.14.MOLDURAS_QUADROS', rf'^(?=.*(MOLDURA))(?=.*(QUADRO|ESPELHO|FOTO))(?=.*(MADEIRA)).*'),
    ('44.15.CAIXOTES_PALETES', rf'^(?=.*(PALETE|PALLET|ESTRADO|CAIXOTE|ENGRADADO))(?!.*({EXCL_SERVICOS})).*'),
    ('44.16.BARRIS_TANOEIRO', r'^(?=.*(BARRIL|TONEL|ADUELA|CUBA|DORNA))(?=.*(MADEIRA)).*'),
    ('44.17.FERRAMENTAS_CABOS', r'^(?=.*(FERRAMENTA|CABO|PÁ|ENXADA|VASSOURA))(?=.*(MADEIRA)).*'),

    # G. NCM 44.18 (Marcenaria Construção)
    ('44.18.10.JANELAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(JANELA|PORTA JANELA))(?=.*(MADEIRA|{TERMOS_ESPECIES})).*'),
    ('44.18.20.PORTAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(PORTA|FOLHA.*PORTA|BATENTE|CAIXILHO))(?!.*(JANELA)).*'),
    ('44.18.40.COFRAGEM', r'^(?=.*(COFRAGEM|FORMA.*CONCRETO))(?=.*(MADEIRA)).*'),
    ('44.18.7X.PISOS_ASSOALHOS', rf'^(?=.*(PISO|ASSOALHO|DECK|SOALHO|TACO))(?=.*(MADEIRA|BAMBU|{TERMOS_ESPECIES})).*'),
    ('44.18.9X.OUTROS_CONSTRUCAO', r'^(?=.*(VIGA|TRELIÇA|FORRO))(?=.*(CONSTRUCAO|OBRA))(?=.*(MADEIRA)).*'),

    # H. NCM 44.19 a 44.21 (Artefatos e Utensílios)
    ('44.19.UTENSILIOS_MESA', r'^(?=.*(PRATO|COPO|TIGELA|COLHER|BANDEJA|TABUA.*CARNE))(?=.*(MADEIRA|BAMBU)).*'),
    ('44.20.MARCHETARIA_ESTATUETAS', rf'^(?!.*({EXCL_DECORACAO}))(?=.*(MARCHETARIA|ESTATUETA|COFRE|CAIXA.*JOIA))(?=.*(MADEIRA)).*'),
    ('44.21.OUTRAS_OBRAS', rf'^(?!.*({EXCL_MATERIAIS}))(?=.*(CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA))(?=.*(MADEIRA|BAMBU)).*'),

    # I. CATEGORIA DE REVISÃO E FECHAMENTO
    ('AMBIGUOS_REVISAR', rf'.*(PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTROS', r'.*')
]

# 3. PROCESSAMENTO E EXPORTAÇÃO
def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

categorias_cap_44 = [c[0] for c in categorias_regex if c[0].startswith('44.') or c[0] == 'ROTULADOS ANTES']
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'AMBIGUOS_REVISAR']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_capitulo_44_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44 (Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                 {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS:             {len(df_eliminados):>10,}".replace(",", "."))


1. CAPÍTULO 44 (Granular):    819.493
2. OUTROS:                    923.425
3. ELIMINADOS:                 12.457

In [ ]:
# Sugestões de configuração com uso de Copilot

'''import re

def tem_madeira_valida(desc: str) -> bool:
    """
    Retorna True se houver evidência clara de madeira ou derivados válidos
    """
    return bool(re.search(
        r'(MADEIRA|MAD\.?|PINUS|EUCALIPTO|BAMBU|'
        r'MDF|HDF|OSB|MDP|COMPENSADO|LAMINADO|'
        r'TABUA|PRANCHA|VIGA|CAIBRO|RIPA|'
        r'TACO|ASSOALHO|DECK|PAINEL)',
        desc
    ))


def tem_exclusao_material(desc: str) -> bool:
    """
    Retorna True se houver material incompatível com Capítulo 44
    """
    return bool(re.search(
        r'(ALUMINIO|FERRO|AÇO|ACO|METAL|INOX|'
        r'PVC|PLASTICO|VIDRO|ACRILICO|'
        r'CERAMICA|PORCELANA|CONCRETO|'
        r'CIMENTO|GRANITO|MARMORE|PEDRA)',
        desc
    ))

def categorizar_produto_fiscal(desc):
    desc = str(desc).upper()

    if re.search(rf'({EXCL_SERVICOS}|LOTE MISTO|MERCADORIA MISTA)', desc):
        return 'OUTLIER__FINANCEIRO'
    if re.search(r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO)', desc):
        return 'OUTLIER__MAQUINAS'
    if re.search(r'(PLANEJADA|COZINHA|DALMOBILE)', desc):
        return 'AMBIGUOS_REVISAR'

    if re.search(
        r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|'
        r'CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|'
        r'PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|'
        r'CHAPAS DE TAPUME|RESIDUO DE MADEIRA)',
        desc
    ):
        return 'ROTULADOS ANTES'

    if not tem_madeira_valida(desc):
        if re.search(r'(PORTA|JANELA|TABUA|VIGA|PAINEL|MDF|PISO|DECK|CAIBRO)', desc):
            return 'ELIMINADO_POR_MATERIAL'
        return 'OUTROS'

    if tem_exclusao_material(desc):
        return 'ELIMINADO_POR_MATERIAL'

    # 44.01 a 44.06
    if re.search(r'(LENHA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA)', desc):
        return '44.01.LENHA_RESIDUOS'
    if re.search(r'(CARVAO|CARVÃO|MOINHA)', desc):
        return '44.02.CARVAO_VEGETAL'
    if re.search(r'(MADEIRA.*(BRUTA|EM BRUTO)|TORA|TORO|RONCO)', desc) and not re.search(
        r'(SERRADA|APARELHADA|BENEFICIADA)', desc
    ):
        return '44.03.MADEIRA_BRUTA'
    if re.search(r'(ARCO|ESTACA|MOIRAO|POSTE)', desc):
        return '44.04.ARCOS_ESTACAS'
    if re.search(r'(LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA)', desc):
        return '44.05.LA_FARINHA'
    if re.search(r'(DORMENTE)', desc):
        return '44.06.DORMENTES'

    # 44.07 a 44.09
    if re.search(r'(TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE)', desc):
        return '44.07.MADEIRA_SERRADA'
    if re.search(r'(FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA)', desc):
        return '44.08.FOLHAS_FOLHEADOS'
    if re.search(r'(MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR)', desc):
        return '44.09.PERFILADOS'

    # 44.10 a 44.12
    if re.search(r'(OSB|MDP|PARTICULA|AGLOMERADO)', desc):
        return '44.10.PAINEIS_PARTICULAS_OSB'
    if re.search(r'(MDF|HDF|FIBRA|ARAUCO|GREENPLAC)', desc):
        return '44.11.PAINEIS_FIBRAS_MDF'
    if re.search(r'(COMPENSADO|CONTRAPLACADO|LAMINADO)', desc):
        return '44.12.COMPENSADO'

    # 44.13 a 44.17
    if re.search(r'(DENSIFICADA|ESTABILIZADA)', desc):
        return '44.13.MADEIRA_DENSIFICADA'
    if re.search(r'(MOLDURA.*(QUADRO|ESPELHO|FOTO))', desc):
        return '44.14.MOLDURAS_QUADROS'
    if re.search(r'(PALETE|PALLET|ESTRADO|CAIXOTE|ENGRADADO)', desc):
        return '44.15.CAIXOTES_PALETES'
    if re.search(r'(BARRIL|TONEL|ADUELA|CUBA|DORNA)', desc):
        return '44.16.BARRIS_TANOEIRO'
    if re.search(r'(FERRAMENTA|CABO|PÁ|ENXADA|VASSOURA)', desc):
        return '44.17.FERRAMENTAS_CABOS'

    # 44.18
    if re.search(r'(PORTA JANELA|JANELA)', desc):
        return '44.18.10.JANELAS'
    if re.search(r'(PORTA|FOLHA.*PORTA|BATENTE|CAIXILHO)', desc):
        return '44.18.20.PORTAS'
    if re.search(r'(COFRAGEM|FORMA.*CONCRETO)', desc):
        return '44.18.40.COFRAGEM'
    if re.search(r'(PISO|ASSOALHO|DECK|SOALHO|TACO)', desc):
        return '44.18.7X.PISOS_ASSOALHOS'

    # 44.19 a 44.21
    if re.search(r'(PRATO|COPO|TIGELA|COLHER|BANDEJA|TABUA.*CARNE)', desc):
        return '44.19.UTENSILIOS_MESA'
    if re.search(r'(MARCHETARIA|ESTATUETA|COFRE|CAIXA.*JOIA)', desc):
        return '44.20.MARCHETARIA_ESTATUETAS'
    if re.search(r'(CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA)', desc):
        return '44.21.OUTRAS_OBRAS'

    return 'OUTROS'

df_pandas['valor_total_vuntrib'] = (
    df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
)

df_pandas['categorias_total'] = df_pandas['prod_xprod'].apply(categorizar_produto_fiscal)

categorias_eliminadas = [
    'OUTLIER__FINANCEIRO',
    'OUTLIER__MAQUINAS',
    'AMBIGUOS_REVISAR',
    'ELIMINADO_POR_MATERIAL'
]

df_cap_44 = df_pandas[
    df_pandas['categorias_total'].str.startswith('44.')
    | (df_pandas['categorias_total'] == 'ROTULADOS ANTES')
].copy()

df_outros = df_pandas[df_pandas['categorias_total'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categorias_total'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_capitulo_44_pre_teste.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre_teste.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre_teste.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44 (Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                 {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS:             {len(df_eliminados):>10,}".replace(",", "."))
'''

# Versão atual

1. CAPÍTULO 44 (Granular):    961.886
2. OUTROS:                    778.126
3. ELIMINADOS:                 13.115

In [ ]:
# 1. DICIONÁRIOS DE TERMOS E EXCLUSÕES

EXCL_MATERIAIS = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO|GRANITO|MARMORE|PEDRA'
EXCL_MOVEIS = 'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA|COZINHA|PLANEJADA|PLANEJADOS'
EXCL_DECORACAO = 'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA|APLIQUE|RECORTE|LASER'
EXCL_SERVICOS = 'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|FRETE|MONTAGEM|INSTALACAO|TAXA|ENTREGA|AJUSTE|BALDEIO|USOS'
TERMOS_ESPECIES = r'\b(?:CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA|IPÊ|IPE|JATOBÁ|JATOBA|SUCUPIRA|ASTRONIUN|ASTRONIUM|CLARISIA|RACEMOSA|URUNDEUVA|GUARIUBA|GUARIÚBA|ANGELIN|JEQUITIBA|CANELA|GARAPEIRA|GRANDIS|GIANDUIA|PINHAL|MAD. LEI|ITAUBA)\b'
MADEIRA_SIMILARES = r'\b(?:MAD|MAD.|MADEIRA|MADEIRAS|MADEIR|MADERIA|MADEIRAW|BMADEIRA|MADEIRA1|MADEIRA3|MADEIRA_M3|EMADEIRA|MMADEIRA|MADEIRRA|MADDEIRA|DEMADEIRA|MADEIRAD|MADEIRADE|MADEIDRA|MADIEIRA|MADEIRAREFORÇADA|MADERIRA)\b'
CARVAO_SIMILARES = r'\b(?:CARVÃO|CARVAO|CARVOSUL|CARVOBOXX|CARVA|CARV|CAR|CACAO|CARVAOOLIVEIRA5KG|CARVAODOURADOS5KG|CARVOES|KICARVAO|CARS|CAVAO|CAVÃO)\b'
TERMO_VEGETAL = r'\b(?:VEGETAL|VEG)\b'
MDF_SIMILARES = r'\b(?:MDF|MDF4|MDF2|MDFC|MDFF|MDF05|MDF03|MDF02|MDF23173|MDF23174|MDF10|MDF23157|MDF15|NDF|HFDF|DF|MDX|MDFCRU|MD4|MDS|MD5|MDN|MD3|MDR|MDC|MD8|MDFREPARO|MSDF|MDCN)\b'

# 2. ESTRUTURA DE CATEGORIAS
categorias_regex = [
    # A. BLOQUEIO DE OUTLIERS
    ('OUTLIER__FINANCEIRO', rf'.*({EXCL_SERVICOS}|LOTE MISTO|MERCADORIA MISTA).*'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|GUINDASTE|EIXO|VW|DRILL|JOHN|DEERE)'),
    ('OUTLIER_OUTROS_CAPS', r'(BOVINOS)'),
    
    # B. ROTULADOS ANTES
    ('ROTULADOS ANTES', rf'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|(?:{CARVAO_SIMILARES}) VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),

    # C. NCM 44.01 a 44.06 (Matéria-Prima e Resíduos)
    ('44.01.LENHA_RESIDUOS', r'^(?=.*(LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA|FINOS|OVERSIZE|MARAVALHA|GRANULADO|TORETE|CASCA))(?!.*(ALUMINIO|FERRO|METAL|PVC)).*'),
    ('44.02.CARVAO_VEGETAL_ESTRITO', rf'^(?=.*{CARVAO_SIMILARES})(?=.*{TERMO_VEGETAL})(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE)).*'),
    ('44.02.CARVAO_VEGETAL_GERAL', rf'^(?=.*{CARVAO_SIMILARES})(?!.*(ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE)).*'),
    ('44.03.MADEIRA_BRUTA', rf'^(?=.*(?:(?:{MADEIRA_SIMILARES}).*(BRUTA|EM BRUTO|BRUTO)|TORA|TORO|RONCO))(?!.*(SERRADA|APARELHADA|BENEFICIADA)).*'),
    ('44.04.ARCOS_ESTACAS', rf'^(?=.*(ARCO|ESTACA|MOIRAO|POSTE))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.05.LA_FARINHA', r'^(?=.*(LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA))(?!.*(MINERAL|VIDRO|SINTETICA)).*'),
    ('44.06.DORMENTES', rf'^(?=.*(DORMENTE))(?=.*(?:(?:{MADEIRA_SIMILARES})|FERROVIA|TRILHO))(?!.*(CONCRETO|METAL)).*'),

    # D. NCM 44.07 a 44.09 (Madeira Processada)
    ('44.07.MADEIRA_SERRADA', rf'^(?=.*(?:\b(?:TABUA|PRANCHA|VIGA|VIGOTA|PRANCHÃO|PRANCHAO|PRANCHAS|MADEIRASERRADA|SERRADO|MADEIRA SERRADA|MADEIRA BENEFICIADA|BENEFICIADA|APARELHADA|SERRADA|MADEIRAVIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE)\b|{TERMOS_ESPECIES}))(?!.*(?:COMPENSADO|MDF|PAINEL|URNA|QUADRO|DIFUSOR|PRANCHETA)).*'),
    ('44.08.FOLHAS_FOLHEADOS', r'^(?=.*(FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA))(?=.*(6MM|6 MM|FINA|DELGADA)).*'),
    ('44.09.PERFILADOS', rf'^(?=.*(MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR))(?=.*(?:(?:{MADEIRA_SIMILARES})|PINUS|EUCALIPTO)).*'),

    # E. NCM 44.10 a 44.12 (Painéis e Chapas)
    ('44.10.PAINEIS_PARTICULAS_OSB', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(OSB|MDP|PARTICULA|AGLOMERADO))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.11.PAINEIS_FIBRAS_MDF', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(?:{MDF_SIMILARES}|HDF|FIBRA|ARAUCO|GREENPLAC|CHAPATEX|DURATEX|BERNECK|EUCATEX))(?=.*(?:18MM|15MM|3MM|2F|1A|{MADEIRA_SIMILARES})).*'),
    ('44.12.COMPENSADO', rf'^(?!.*({EXCL_MOVEIS}))(?=.*(COMPENSADO|CONTRAPLACADO|LAMINADO|NAVAL|SARRAFEADO|TAPUME))(?=.*(?:{MADEIRA_SIMILARES})).*'),

    # F. NCM 44.13 a 44.17 (Obras Diversas)
    ('44.13.MADEIRA_DENSIFICADA', rf'^(?=.*(DENSIFICADA|ESTABILIZADA))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.14.MOLDURAS_QUADROS', rf'^(?=.*(MOLDURA))(?=.*(QUADRO|ESPELHO|FOTO))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.15.CAIXOTES_PALETES', rf'^(?=.*(PALETE|PALLET|ESTRADO|CAIXOTE|ENGRADADO|ESTRADO))(?!.*({EXCL_SERVICOS})).*'),
    ('44.16.BARRIS_TANOEIRO', rf'^(?=.*(BARRIL|TONEL|ADUELA|CUBA|DORNA))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.17.FERRAMENTAS_CABOS', rf'^(?=.*(FERRAMENTA|CABO|PÁ|ENXADA|VASSOURA))(?=.*(?:{MADEIRA_SIMILARES})).*'),

    # G. NCM 44.18 (Marcenaria Construção)
    ('44.18.10.JANELAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(JANELA|VENEZIANA|PORTA JANELA))(?=.*(?:(?:{MADEIRA_SIMILARES})|{TERMOS_ESPECIES})).*'),
    ('44.18.20.PORTAS', rf'^(?!.*({EXCL_MATERIAIS}|{EXCL_DECORACAO}))(?=.*(PORTA|FOLHA.*PORTA|BATENTE|CAIXILHO))(?!.*(JANELA)).*'),
    ('44.18.40.COFRAGEM', rf'^(?=.*(COFRAGEM|FORMA.*CONCRETO))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.18.7X.PISOS_ASSOALHOS', rf'^(?=.*(PISO|ASSOALHO|DECK|SOALHO|TACO))(?=.*(?:(?:{MADEIRA_SIMILARES})|BAMBU|{TERMOS_ESPECIES})).*'),
    ('44.18.9X.OUTROS_CONSTRUCAO', rf'^(?=.*(VIGA|TRELIÇA|FORRO))(?=.*(CONSTRUCAO|OBRA))(?=.*(?:{MADEIRA_SIMILARES})).*'),

    # H. NCM 44.19 a 44.21 (Artefatos e Utensílios)
    ('44.19.UTENSILIOS_MESA', rf'^(?=.*(PRATO|COPO|TIGELA|COLHER|BANDEJA|TABUA.*CARNE))(?=.*(?:(?:{MADEIRA_SIMILARES})|BAMBU)).*'),
    ('44.20.MARCHETARIA_ESTATUETAS', rf'^(?!.*({EXCL_DECORACAO}))(?=.*(MARCHETARIA|ESTATUETA|COFRE|CAIXA.*JOIA))(?=.*(?:{MADEIRA_SIMILARES})).*'),
    ('44.21.OUTRAS_OBRAS', rf'^(?!.*({EXCL_MATERIAIS}))(?=.*(CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA|BANDEJA|PETISQUEIRA|TIGELA|CUMBUCA|ESTATUETA|CAIXA|PRENDEDOR|RETRATO))(?=.*(?:(?:{MADEIRA_SIMILARES})|BAMBU)).*'),

    # I. CATEGORIA DE REVISÃO E FECHAMENTO
    ('AMBIGUOS_REVISAR', rf'.*(PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA).*'),
    ('GUARNICAO', rf'.*(GUARNICAO|GUARNIÇÃO|GUARNIÇAO|GUARNICÃO).*'),
    ('OUTROS', r'.*')
]

# 3. PROCESSAMENTO E EXPORTAÇÃO
def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

categorias_cap_44 = [c[0] for c in categorias_regex if c[0].startswith('44.') or c[0] == 'ROTULADOS ANTES']
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'AMBIGUOS_REVISAR']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_capitulo_44_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar_pre.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo_pre.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44 (Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                 {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS:             {len(df_eliminados):>10,}".replace(",", "."))
